# 07: Unified Model Backtest & Performance Diagnostics Dashboard

This interactive research notebook provides a **unified, multi-lens backtest performance evaluation** across all 3 active quantitative models in the **MDK Trading Oracle**:
- **Model 1: Macro Day-Start Forecaster** (Exchange-wide opening net flow in Million TL)
- **Model 2: Sector Day-Start Forecaster** (Cross-sector capital allocation across 26 BIST sectors in Million TL)
- **Model 3: Stock Intraday Reaction Forecaster** (Intraday stock return percentages across W2, W3, and W5 windows for BIST 30 equities)

---

### Four Complementary Evaluation Lenses
1. **Continuous Regression Precision**: MAE, RMSE, Pearson Correlation ($r$), $R^2$, and Residual Skew/Bias.
2. **Directional & Conviction Monotonicity**: Sign Hit Rate %, Conviction Tier breakdown (`STRONG_BUY` $\to$ `STRONG_SELL`), verifying that higher conviction yields higher win rates.
3. **Probabilistic Calibration**: 90% Prediction Interval Coverage Probability (PICP 90%) and interval width (MPIW sharpness).
4. **Trading Utility & Temporal Drift**: Directional strategy capture proxy, win/loss ratio, profit factor, and cumulative accuracy stability.


## 1. Setup & Loader Initialization

We connect to the local DuckDB database strictly in **read-only mode** (`read_only=True`) and initialize the unified `BacktestLoader`.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

from mdk_trading_oracle.core.db import DuckDBManager
from mdk_trading_oracle.backtest import (
    BacktestLoader,
    BacktestMetricsCalculator,
    BacktestVisualizer,
    BacktestReportGenerator,
    TargetUnit,
)

# Connect to DuckDB via read-only loader
db = DuckDBManager(read_only=True)
loader = BacktestLoader(db=db)
print("Lakehouse connection established (read-only mode). BacktestLoader ready.")


## 2. Executive Multi-Model Benchmark

We load backtest datasets across all 3 models from DuckDB Gold tables (`gold_bofa_*_backtests`), calculate statistical performance summaries, and present a side-by-side comparative leaderboard.


In [ ]:
# Load summaries for all three models
summary_m1 = loader.summarize_day_start()
summary_m2 = loader.summarize_sector_day_start()
summary_m3 = loader.summarize_stock_reaction(windows=["w2", "w3", "w5"])

# Generate executive side-by-side benchmark table
df_benchmark = BacktestReportGenerator.compare_models([summary_m1, summary_m2, summary_m3])
display(df_benchmark)


## 3. Model 1: Macro Day-Start Forecaster Deep-Dive

Evaluating BofA exchange-wide opening net flow forecasts ($TL$).


In [ ]:
# Executive KPI Card for Model 1
card_html = BacktestVisualizer.format_executive_scorecard_html(summary_m1)
display(HTML(card_html))

# Load full Model 1 backtest dataframe
df_m1 = loader.load_day_start()

# Chart 1: Actual vs Predicted Time-Series Track Record with 90% Confidence Ribbon
fig_track_m1 = BacktestVisualizer.plot_track_record(
    df=df_m1,
    summary=summary_m1,
    unit=TargetUnit.TL,
    title="Model 1: BofA Day-Start Macro Flow Track Record (Predicted vs Realized with 90% Credible Ribbon)",
)
fig_track_m1.show()


In [ ]:
# Chart 2: Parity Scatter & Residual Error Distribution
fig_parity_m1 = BacktestVisualizer.plot_parity_and_residuals(
    df=df_m1,
    summary=summary_m1,
    unit=TargetUnit.TL,
    title="Model 1: Parity Diagnostics & Error Distribution (Residuals = Predicted - Actual)",
)
fig_parity_m1.show()


In [ ]:
# Chart 3: Cumulative Performance & Temporal Drift
fig_cum_m1 = BacktestVisualizer.plot_cumulative_performance(
    df=df_m1,
    summary=summary_m1,
    unit=TargetUnit.TL,
    title="Model 1: Cumulative Directional Strategy Capture & Expanding Hit Rate Drift",
)
fig_cum_m1.show()


In [ ]:
# Chart 4: Conviction Tier Monotonicity & Probabilistic Calibration
fig_calib_m1 = BacktestVisualizer.plot_conviction_and_calibration(
    summary=summary_m1,
    title="Model 1: Conviction Ladder Win Rates & 90% Credible Interval Coverage",
)
fig_calib_m1.show()


In [ ]:
# Slice Analysis: Model 1 Performance by Predicted Execution Playbook
if "predicted_playbook" in summary_m1.slices:
    fig_playbook = BacktestVisualizer.plot_slice_leaderboard(
        slice_metrics=summary_m1.slices["predicted_playbook"],
        title="Model 1: Hit Rate % by Institutional Execution Playbook",
        unit=TargetUnit.TL,
    )
    fig_playbook.show()


## 4. Model 2: Sector Day-Start Forecaster & Sector Allocation

Evaluating BofA cross-sector capital rotation forecasts across 26 BIST sectors.


In [ ]:
# Executive KPI Card for Model 2
card_html_m2 = BacktestVisualizer.format_executive_scorecard_html(summary_m2)
display(HTML(card_html_m2))

# Load full Model 2 sector backtest dataframe
df_m2 = loader.load_sector_day_start()

# Chart: Cross-Sector Leaderboard (Ranked by Out-of-Sample Hit Rate)
if "sector" in summary_m2.slices:
    fig_sector_lead = BacktestVisualizer.plot_slice_leaderboard(
        slice_metrics=summary_m2.slices["sector"],
        title="Model 2: Cross-Sector Backtest Leaderboard (Ranked by Out-of-Sample Directional Hit Rate)",
        max_display=26,
        unit=TargetUnit.TL,
    )
    fig_sector_lead.show()


In [ ]:
# Interactive Single-Sector Drilldown Inspector
sector_list = sorted(df_m2["sector"].unique().tolist())
sector_dropdown = widgets.Dropdown(
    options=sector_list,
    value="Banking" if "Banking" in sector_list else sector_list[0],
    description="Sector:",
    style={"description_width": "initial"}
)

output_sector_drilldown = widgets.Output()

def update_sector_drilldown(change):
    with output_sector_drilldown:
        output_sector_drilldown.clear_output()
        sel = change["new"]
        sub_df = df_m2[df_m2["sector"] == sel].copy()
        if sub_df.empty:
            print(f"No records found for {sel}.")
            return
        sub_summary = BacktestMetricsCalculator.calculate_summary(
            df=sub_df,
            model_name=f"Model 2 — Sector: {sel}",
            target_unit=TargetUnit.TL,
        )
        fig = BacktestVisualizer.plot_track_record(
            df=sub_df,
            summary=sub_summary,
            unit=TargetUnit.TL,
            title=f"BofA Window 1 Opening Flow: {sel} (Backtest vs Actual)",
        )
        fig.show()

sector_dropdown.observe(update_sector_drilldown, names="value")
display(sector_dropdown, output_sector_drilldown)
update_sector_drilldown({"new": sector_dropdown.value})


## 5. Model 3: Stock Intraday Reaction Forecaster Deep-Dive

Evaluating stock percentage returns (`%`) across reaction windows:
- `W2` (first_reaction, 10:30-11:30)
- `W3` (midday_followup, 11:30-14:30)
- `W5` (closing_session, 16:00-18:15)


In [ ]:
# Executive KPI Card for Model 3
card_html_m3 = BacktestVisualizer.format_executive_scorecard_html(summary_m3)
display(HTML(card_html_m3))

# Load full Model 3 backtest dataframe
df_m3 = loader.load_stock_reaction()

# Sub-segment breakdown across intraday reaction windows
if "window_name" in summary_m3.slices:
    df_windows = pd.DataFrame([
        {
            "Window": s.slice_key.upper(),
            "Samples": s.sample_count,
            "Hit Rate (%)": round(s.hit_rate_pct, 1),
            "MAE (%)": round(s.mae, 2),
            "RMSE (%)": round(s.rmse, 2),
            "PICP 90% (%)": round(s.picp_90_pct, 1),
            "Directional Capture (%)": round(s.directional_capture, 2),
        }
        for s in summary_m3.slices["window_name"]
    ])
    print("Intraday Reaction Window Benchmark:")
    display(df_windows)


In [ ]:
# Chart: Stock x Window Heatmap Matrix
fig_matrix_m3 = BacktestVisualizer.plot_stock_window_matrix(
    df=df_m3,
    title="Model 3: Stock Intraday Directional Hit Rate Matrix (Symbol x Window)",
    symbol_col="symbol",
    window_col="window_name",
    is_hit_col="is_direction_hit",
)
fig_matrix_m3.show()


In [ ]:
# Interactive Single-Stock Drilldown Inspector
stock_list = sorted(df_m3["symbol"].unique().tolist())
stock_dropdown = widgets.Dropdown(
    options=stock_list,
    value="THYAO" if "THYAO" in stock_list else stock_list[0],
    description="Symbol:",
    style={"description_width": "initial"}
)

window_dropdown = widgets.Dropdown(
    options=[
        ("W2: First Reaction (10:30-11:30)", "first_reaction"),
        ("W3: Midday Follow-up (11:30-14:30)", "midday_followup"),
        ("W5: Closing Session (16:00-18:15)", "closing_session"),
    ],
    value="first_reaction",
    description="Window:",
    style={"description_width": "initial"}
)

output_stock_drilldown = widgets.Output()

def update_stock_drilldown(*args):
    with output_stock_drilldown:
        output_stock_drilldown.clear_output()
        sym = stock_dropdown.value
        win = window_dropdown.value
        win_map = {
            "w2": "first_reaction",
            "w3": "midday_followup",
            "w5": "closing_session",
            "first_reaction": "first_reaction",
            "midday_followup": "midday_followup",
            "closing_session": "closing_session",
        }
        canonical_win = win_map.get(str(win).lower(), win)
        sub_df = df_m3[(df_m3["symbol"] == sym) & (df_m3["window_name"] == canonical_win)].copy()
        if sub_df.empty:
            print(f"No records found for {sym} {win}.")
            return
        disp_map = {
            "first_reaction": "W2 (10:30-11:30)",
            "midday_followup": "W3 (11:30-14:30)",
            "closing_session": "W5 (16:00-18:15)",
        }
        win_title = disp_map.get(canonical_win, canonical_win.upper())
        sub_summary = BacktestMetricsCalculator.calculate_summary(
            df=sub_df,
            model_name=f"Model 3 — {sym} ({win_title})",
            actual_col="actual_return_pct",
            predicted_col="predicted_return_pct",
            lower_90_col="predicted_return_lower_90",
            upper_90_col="predicted_return_upper_90",
            target_unit=TargetUnit.PERCENTAGE,
        )
        fig = BacktestVisualizer.plot_track_record(
            df=sub_df,
            summary=sub_summary,
            actual_col="actual_return_pct",
            predicted_col="predicted_return_pct",
            lower_90_col="predicted_return_lower_90",
            upper_90_col="predicted_return_upper_90",
            unit=TargetUnit.PERCENTAGE,
            title=f"Stock Reaction: {sym} ({win_title}) — Actual vs Predicted Return %",
        )
        fig.show()

stock_dropdown.observe(update_stock_drilldown, names="value")
window_dropdown.observe(update_stock_drilldown, names="value")
display(widgets.HBox([stock_dropdown, window_dropdown]), output_stock_drilldown)
update_stock_drilldown()


## 6. Comprehensive Markdown Audit Report Export

Exporting a structured markdown summary report suitable for distribution and audit logging.


In [ ]:
# Generate and print full markdown audit report for Model 1
report_md = BacktestReportGenerator.generate_markdown_summary(summary_m1)
display(Markdown(report_md))
